# Gráficos das atividades 2, 3 e 4

Os gráficos leem os arquivos de novo. Não dependem de variáveis deixadas em outro notebook.

Na atividade 2, homens e mulheres são a média simples dos recortes branca e preta ou parda. Isso segue o exemplo da aula e não é uma média oficial ponderada.

Execute a partir da pasta da atividade ou da raiz de `analise_dados`.


In [1]:
import collections
import collections.abc

for nome in ("Iterable", "Mapping", "Sequence"):
    if not hasattr(collections, nome):
        setattr(collections, nome, getattr(collections.abc, nome))

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import xlrd

def ler_xls(caminho, aba):
    # xlrd 1.2 abre .xls. O pandas recente pede xlrd 2, que não lê esse formato.
    livro = xlrd.open_workbook(str(caminho))
    planilha = livro.sheet_by_name(aba)
    linhas = [planilha.row_values(i) for i in range(planilha.nrows)]
    return pd.DataFrame(linhas)

def raiz_do_repositorio():
    aqui = Path.cwd().resolve()
    for pasta in [aqui, *aqui.parents]:
        if (pasta / "dados").is_dir() and (pasta / "etapa_3").is_dir():
            return pasta
    raise FileNotFoundError("Execute a partir do repositorio analise_dados.")

RAIZ = raiz_do_repositorio()
DADOS = RAIZ / "dados"
SAIDAS = RAIZ / "etapa_3" / "04_graficos" / "saidas"
SAIDAS.mkdir(parents=True, exist_ok=True)


## Atividade 2 — horas por região e sexo


In [2]:
bruto_horas = ler_xls(DADOS / "Tabela 1.1.1.xls", "2022")
indicador_1 = bruto_horas.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)
for coluna in indicador_1.columns[1:]:
    indicador_1[coluna] = pd.to_numeric(indicador_1[coluna], errors="coerce")

regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
reg = indicador_1[indicador_1["uf_regiao"].isin(regioes)].copy()
reg["homens"] = (reg["homem_branca"] + reg["homem_preta_parda"]) / 2
reg["mulheres"] = (reg["mulher_branca"] + reg["mulher_preta_parda"]) / 2

reg_long = reg.melt(
    id_vars="uf_regiao",
    value_vars=["homens", "mulheres"],
    var_name="sexo",
    value_name="horas",
)
reg_long


,uf_regiao,sexo,horas
0,Norte,homens,11.551649
1,Nordeste,homens,11.786171
2,Sudeste,homens,12.057182
3,Sul,homens,11.353058
4,Centro-Oeste,homens,10.439429
5,Norte,mulheres,19.914984
6,Nordeste,mulheres,23.230691
7,Sudeste,mulheres,21.361700
8,Sul,mulheres,19.175851
9,Centro-Oeste,mulheres,18.775862


In [3]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=reg_long, x="uf_regiao", y="horas", hue="sexo", ax=ax)
ax.set_title("Afazeres domésticos por região e sexo")
ax.set_xlabel("Região")
ax.set_ylabel("Horas / semana")
ax.legend(title="Sexo")
plt.tight_layout()
fig.savefig(SAIDAS / "atividade2_horas_regiao.png", dpi=120, bbox_inches="tight")
plt.show()


C:\Users\alzat\AppData\Local\Temp\ipykernel_15764\628465244.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Atividade 3 — graduação no Brasil

Os nomes `superior`, `ctem` e `educacao_saude` são os da análise já entregue. O gráfico usa o mesmo recorte da planilha.


In [4]:
bruto_formacao = pd.read_excel(
    DADOS / "Tabela_3_Areas_gerais_de_formacao_na_graduacao.xlsx",
    sheet_name="Sheet1",
    header=None,
)

superior = bruto_formacao.iloc[3:36, [0, 1, 2, 3]].copy()
superior.columns = ["regiao", "total", "homens", "mulheres"]
ctem = bruto_formacao.iloc[3:36, [0, 4, 5, 6]].copy()
ctem.columns = ["regiao", "total", "homens", "mulheres"]
educacao_saude = bruto_formacao.iloc[3:36, [0, 7, 8, 9]].copy()
educacao_saude.columns = ["regiao", "total", "homens", "mulheres"]

for quadro in (superior, ctem, educacao_saude):
    quadro.reset_index(drop=True, inplace=True)
    for coluna in quadro.columns[1:]:
        quadro[coluna] = pd.to_numeric(quadro[coluna], errors="coerce")

brasil = pd.DataFrame(
    {
        "area": ["Graduação (total)", "STEM", "Saúde / educação"],
        "homens": [
            superior.loc[superior["regiao"] == "Brasil", "homens"].item(),
            ctem.loc[ctem["regiao"] == "Brasil", "homens"].item(),
            educacao_saude.loc[educacao_saude["regiao"] == "Brasil", "homens"].item(),
        ],
        "mulheres": [
            superior.loc[superior["regiao"] == "Brasil", "mulheres"].item(),
            ctem.loc[ctem["regiao"] == "Brasil", "mulheres"].item(),
            educacao_saude.loc[educacao_saude["regiao"] == "Brasil", "mulheres"].item(),
        ],
    }
)
brasil_long = brasil.melt(id_vars="area", var_name="sexo", value_name="pessoas")
brasil


C:\Users\alzat\AppData\Local\Temp\ipykernel_15764\3577480122.py:15: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quadro.reset_index(drop=True, inplace=True)
C:\Users\alzat\AppData\Local\Temp\ipykernel_15764\3577480122.py:15: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quadro.reset_index(drop=True, inplace=True)
C:\Users\alzat\AppData\Local\Temp\ipykernel_15764\3577480122.py:15: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quadro.reset_index(drop=True, inp

,area,homens,mulheres
0,Graduação (total),10478534,15375757
1,STEM,2750422,1398619
2,Saúde / educação,1677985,6401723


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=brasil_long, x="area", y="pessoas", hue="sexo", ax=ax)
ax.set_title("Brasil: pessoas com graduação, por área e sexo")
ax.set_xlabel("Área")
ax.set_ylabel("Pessoas")
ax.legend(title="Sexo")
plt.tight_layout()
fig.savefig(SAIDAS / "atividade3_graduacao_brasil.png", dpi=120, bbox_inches="tight")
plt.show()


C:\Users\alzat\AppData\Local\Temp\ipykernel_15764\3758488946.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Atividade 4 — população de 2022 por UF

O gráfico ordena pela população de 2022, como no exemplo da aula. A tabela da atividade 4 continua ordenada pelo crescimento.


In [6]:
bruto_pop = pd.read_excel(
    DADOS / "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx",
    sheet_name="Municípios",
    header=None,
)
mascara = bruto_pop.apply(
    lambda coluna: coluna.astype(str).str.contains("NOME DO MUNIC", na=False)
)
linha_cabecalho = int(mascara.any(axis=1).idxmax())
cabecalho = bruto_pop.iloc[linha_cabecalho]
indice_uf = next(i for i, texto in cabecalho.items() if str(texto).strip() == "UF")
indice_2022 = next(i for i, texto in cabecalho.items() if "2022" in str(texto))

siglas = [
    "RO", "AC", "AM", "RR", "PA", "AP", "TO",
    "MA", "PI", "CE", "RN", "PB", "PE", "AL", "SE", "BA",
    "MG", "ES", "RJ", "SP",
    "PR", "SC", "RS",
    "MS", "MT", "GO", "DF",
]
municipios = bruto_pop.iloc[linha_cabecalho + 1 :, [indice_uf, indice_2022]].copy()
municipios.columns = ["UF", "2022"]
municipios = municipios[municipios["UF"].isin(siglas)].copy()
municipios["2022"] = pd.to_numeric(municipios["2022"], errors="coerce")

pop_estado = municipios.groupby("UF", as_index=False)["2022"].sum()
ordem = pop_estado.sort_values("2022", ascending=False)
print("ufs no grafico:", len(ordem))
ordem.head()


ufs no grafico: 27


,UF,2022
25,SP,41262199
10,MG,19597330
18,RJ,15989929
4,BA,14017071
22,RS,10693929


In [7]:
fig, ax = plt.subplots(figsize=(9, 8))
sns.barplot(data=ordem, y="UF", x="2022", ax=ax, color="steelblue", order=ordem["UF"].tolist())
ax.set_title("População 2022 por UF")
ax.set_xlabel("Habitantes")
ax.set_ylabel("UF")
plt.tight_layout()
fig.savefig(SAIDAS / "atividade4_populacao_2022.png", dpi=120, bbox_inches="tight")
plt.show()

C:\Users\alzat\AppData\Local\Temp\ipykernel_15764\3380611928.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
